In [9]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [1]:
words = open('names.txt','r').read().splitlines()


In [2]:
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [3]:
len(words)

32033

In [4]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [134]:
#build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one? Padding with '.' which is mapped to 0
X, Y = [], []
for w in words: # take the first 1000 words to train on
    #print(w)
    context = [0] * block_size # initial context is all '.'
    for ch in w + '.': # for each character in the word + the stop character
        ix = stoi[ch] # get the index of the character
        X.append(context) # add the current context to X
        Y.append(ix) # add the index of the next character to Y
        #print(''.join(itos[i] for i in context), '->', itos[ix]) # print the current context and the next character
        context = context[1:] + [ix] # slide the context window and add the new character
        
X = torch.tensor(X)
Y = torch.tensor(Y)

In [136]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [137]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        ...,
        [26, 26, 25],
        [26, 25, 26],
        [25, 26, 24]])

In [138]:
Y

tensor([ 5, 13, 13,  ..., 26, 24,  0])

In [139]:
#We will cram the 27 possible chartacters into a 2d space, and then learn a linear classifier on top of that
C = torch.rand(27,2) # lookup table with 27 rows anbd 2 columns, initialized randomly


In [140]:
C[5]

tensor([0.3597, 0.3976])

In [141]:
#F.one_hot(5, num_classes=27) This will not work because it returns a 1d tensor, and we need a 2d tensor to do the matrix multiplication with C. We can use unsqueeze to add an extra dimension to the tensor.
F.one_hot(torch.tensor(5), num_classes=27)

tensor([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0])

In [142]:
F.one_hot(Y, num_classes=27).dtype

torch.int64

In [143]:
#If we take the one hot and multiply it by C, we get an error because one hot is a long integer while C is a float, Python does not know how to multiply a long integer by a float, we need to convert the one hot to a float tensor.
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([0.3597, 0.3976])

#since they are the same result from the indexing and usin one hotm we will stick with the indexing method because it is more efficient and easier to read. We can use the one hot method to check our work and make sure we are getting the same result.

#How do we embbed all the 32 ,3 integers stores in array X, we can index with a list or tensor oif integers

In [144]:
C[[5,6,7]]

tensor([[0.3597, 0.3976],
        [0.1435, 0.7920],
        [0.2236, 0.5292]])

In [145]:
C[torch.tensor([[5,6,7]])]

tensor([[[0.3597, 0.3976],
         [0.1435, 0.7920],
         [0.2236, 0.5292]]])

In [146]:
C[torch.tensor([[5,6,7,7,7,7,7,7]])]

tensor([[[0.3597, 0.3976],
         [0.1435, 0.7920],
         [0.2236, 0.5292],
         [0.2236, 0.5292],
         [0.2236, 0.5292],
         [0.2236, 0.5292],
         [0.2236, 0.5292],
         [0.2236, 0.5292]]])

In [147]:
# We can aklso index with a 2d  tensor as following:
C[X].shape

torch.Size([228146, 3, 2])

In [148]:
X[13,2]

tensor(1)

In [149]:
C[X][13,2]

tensor([0.3652, 0.0198])

In [150]:
C[1]

tensor([0.3652, 0.0198])

In [151]:
#So, to embed simultaneously all the integers in X,we can simply do C[X], which will give us a 3d tensor of shape (number of examples, block_size, embedding_size) in our case (number of examples, 3, 2)
emb = C[X]
emb.shape

torch.Size([228146, 3, 2])

In [152]:
w1 = torch.randn((6,100)) # first layer weights
b1 = torch.randn(100) # first layer bias

In [153]:
# Typically we take the input multiply with the weight and add bias, the issue us the embeddings are stacked up in the dimensions of the input tensor.
# How do we transform the 32,3,2 into 32,6 to be abke to calculate the embedding 
# We need to concatenate the inout to be able to do emb @ w1 + b
# We will want to retrieve all the 3 parts in the networks and concetenate them
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], dim=1) # this will give us the three parts of the embedding, we can then concatenate them together to get a 32,6 tensor that we can multiply with w1 and add b1. We can use torch.cat to concatenate the tensors together. We need to specify the dimension to concatenate on, in this case we want to concatenate on the last dimension, which is the embedding dimension. So we will concatenate on dimension 2. We also need to specify the order of the tensors we want to concatenate, in this case we want to concatenate the first part of the embedding, then the second part, then the third part. So we will concatenate emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]].




tensor([[0.3829, 0.1246, 0.3829, 0.1246, 0.3829, 0.1246],
        [0.3829, 0.1246, 0.3829, 0.1246, 0.3597, 0.3976],
        [0.3829, 0.1246, 0.3597, 0.3976, 0.6263, 0.5465],
        ...,
        [0.3323, 0.2982, 0.3323, 0.2982, 0.6497, 0.5552],
        [0.3323, 0.2982, 0.6497, 0.5552, 0.3323, 0.2982],
        [0.6497, 0.5552, 0.3323, 0.2982, 0.7396, 0.0159]])

In [154]:
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], dim=1).shape
#Since we do not want to keeop chaning when we modify the context/block soize, we will need to use unbind to remove a dimension

torch.Size([228146, 6])

In [155]:
torch.unbind(emb, dim=1) # this will give us a list of 3 tensors, each of shape (number of examples, embedding_size), we can then concatenate them together to get a tensor of shape (number of examples, block_size * embedding_size) that we can multiply with w1 and add b1. We can use torch.cat to concatenate the tensors together. We need to specify the dimension to concatenate on, in this case we want to concatenate on the last dimension, which is the embedding dimension. So we will concatenate on dimension 1. We also need to specify the order of the tensors we want to concatenate, in this case we want to concatenate the first part of the embedding, then the second part, then the third part. So we will concatenate emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]].

(tensor([[0.3829, 0.1246],
         [0.3829, 0.1246],
         [0.3829, 0.1246],
         ...,
         [0.3323, 0.2982],
         [0.3323, 0.2982],
         [0.6497, 0.5552]]),
 tensor([[0.3829, 0.1246],
         [0.3829, 0.1246],
         [0.3597, 0.3976],
         ...,
         [0.3323, 0.2982],
         [0.6497, 0.5552],
         [0.3323, 0.2982]]),
 tensor([[0.3829, 0.1246],
         [0.3597, 0.3976],
         [0.6263, 0.5465],
         ...,
         [0.6497, 0.5552],
         [0.3323, 0.2982],
         [0.7396, 0.0159]]))

In [156]:
torch.cat(torch.unbind(emb, dim=1), dim=1).shape  #does not matter blok size with this code
# Using torch,cat is inefficient becasuse new memory is created, creates a whole new tensor with new storage, cannot just manipoulate the view attributes.

torch.Size([228146, 6])

In [157]:
a = torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [158]:
a.view(3,3,2)

tensor([[[ 0,  1],
         [ 2,  3],
         [ 4,  5]],

        [[ 6,  7],
         [ 8,  9],
         [10, 11]],

        [[12, 13],
         [14, 15],
         [16, 17]]])

In [159]:
a.storage() #eah tensotr has a storage which is a 1d array that contains all the data of the tensor, and the tensor itself is just a view on that storage with a certain shape and stride. The storage is what is actually stored in memory, and the tensor is just a way to interpret that data. The view method allows us to change the shape of the tensor without changing the underlying data, while the storage method allows us to access the underlying data directly.

 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
 12
 13
 14
 15
 16
 17
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 18]

In [160]:
emb.view(32,6)

RuntimeError: shape '[32, 6]' is invalid for input of size 1368876

In [161]:
emb.view(32,6) == torch.cat(torch.unbind(emb, dim=1), dim=1) # These 2 are equal

RuntimeError: shape '[32, 6]' is invalid for input of size 1368876

In [162]:
h = emb.view(emb.shape[0],6) @ w1 + b1 # this is the same as torch.cat(torch.unbind(emb, dim=1), dim=1) @ w1 + b1
# Can also use h = emb.view(-1, 6) @ w1 + b1, the -1 will automatically infer the correct size for that dimension based on the other dimensions and the total number of elements in the tensor. This is a convenient way to reshape a tensor when we don't want to manually calculate the size of one of the dimensions.

In [163]:
h

tensor([[ 1.1235,  1.3689, -0.3890,  ...,  1.4780, -0.6717,  1.3777],
        [ 0.6633,  1.7186, -0.2558,  ...,  1.1437, -0.8257,  1.2975],
        [ 0.5808,  1.5590,  0.2152,  ...,  0.9439, -0.9165,  1.5725],
        ...,
        [ 0.6457,  1.9695,  0.0538,  ...,  0.8324, -0.6638,  1.2291],
        [ 1.7095,  1.3060,  0.2460,  ...,  2.2949, -0.6199,  2.1164],
        [ 1.9769,  1.5771, -0.0723,  ...,  1.9649, -0.5232,  0.7388]])

In [164]:
# Most efficient way
h = torch.tanh(emb.view(-1, 6) @ w1 + b1) # this is the same as torch.tanh(torch.cat(torch.unbind(emb, dim=1), dim=1) @ w1 + b1) but more efficient because it does not create a new tensor with new storage, it just manipulates the view attributes of the original tensor.

In [165]:
h

tensor([[ 0.8088,  0.8784, -0.3705,  ...,  0.9011, -0.5861,  0.8804],
        [ 0.5806,  0.9377, -0.2503,  ...,  0.8156, -0.6782,  0.8611],
        [ 0.5233,  0.9153,  0.2119,  ...,  0.7370, -0.7243,  0.9174],
        ...,
        [ 0.5687,  0.9618,  0.0537,  ...,  0.6818, -0.5809,  0.8423],
        [ 0.9366,  0.8633,  0.2411,  ...,  0.9799, -0.5511,  0.9714],
        [ 0.9624,  0.9182, -0.0722,  ...,  0.9615, -0.4802,  0.6284]])

In [166]:
h.shape

torch.Size([228146, 100])

In [167]:
#Broadcasting check
(emb.view(-1, 6) @ w1).shape

torch.Size([228146, 100])

In [168]:
b1.shape

torch.Size([100])

In [169]:
#32,100
# 1,100, broadcasting will align on the right and take a fake dimension on the left
#Broadcasting will copy vertically all the rows of 32 and do an elementwise addition, sanme bias vector will be added to all the rows, whihc is correct.

In [170]:
h.shape

torch.Size([228146, 100])

In [171]:
w2 = torch.randn((100,27)) # second layer weights
b2 = torch.randn((27)) # second layer biases

In [172]:
logits = h @ w2 + b2 # this will give us the logits for each of the 27 possible characters, we can then use these logits to calculate the loss and do backpropagation to update the weights and biases of the network. The shape of logits will be (number of examples, number of classes), in this case (number of examples, 27) because we have 27 possible characters to predict. We can then use these logits to calculate the loss and do backpropagation to update the weights and biases of the network.

In [173]:
logits.shape

torch.Size([228146, 27])

In [174]:
counts = logits.exp() # this will give us the counts for each of the 27 possible characters, we can then use these counts to calculate the probabilities and do backpropagation to update the weights and biases of the network. The shape of counts will be (number of examples, number of classes), in this case (number of examples, 27) because we have 27 possible characters to predict. We can then use these counts to calculate the probabilities and do backpropagation to update the weights and biases of the network.

In [175]:
prob = counts/counts.sum(1, keepdim=True) # this will give us the probabilities for each of the 27 possible characters, we can then use these probabilities to calculate the loss and do backpropagation to update the weights and biases of the network. The shape of prob will be (number of examples, number of classes), in this case (number of examples, 27) because we have 27 possible characters to predict. We can then use these probabilities to calculate the loss and do backpropagation to update the weights and biases of the network. We need to sum the counts along the second dimension (the classes) to get the total count for each example, and then we need to keep the dimensions of the result so that we can divide the counts by the total count for each example to get the probabilities. This is done by setting keepdim=True in the sum function.

In [176]:
prob.shape

torch.Size([228146, 27])

In [177]:
prob[0].sum() # the probabilities for each example should sum to 1, we can check this by summing the probabilities for the first example and see if it equals 1. This is a good way to check if our probabilities are correct, if they do not sum to 1 then there is likely a bug in our code that we need to fix.

tensor(1.0000)

In [178]:
Y 

tensor([ 5, 13, 13,  ..., 26, 24,  0])

In [179]:
torch.arange(32)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

In [182]:
prob[torch.arange(32), Y] # this will give us the probabilities for the correct characters for each example, we can then use these probabilities to calculate the loss and do backpropagation to update the weights and biases of the network. The shape of this tensor will be (number of examples,) because we are selecting one probability for each example, which corresponds to the correct character that we are trying to predict. We can then use these probabilities to calculate the loss and do backpropagation to update the weights and biases of the network.    

IndexError: shape mismatch: indexing tensors could not be broadcast together with shapes [32], [228146]

In [183]:
loss = -prob[torch.arange(32), Y].log().mean() # this will give us the average negative log likelihood loss for the correct characters, we can then use this loss to do backpropagation to update the weights and biases of the network. The shape of this tensor will be a single scalar value because we are taking the mean of the negative log probabilities for all the examples. We can then use this loss to do backpropagation to update the weights and biases of the network.
loss

IndexError: shape mismatch: indexing tensors could not be broadcast together with shapes [32], [228146]

#Cleaning up the Code

In [184]:
X.shape,Y.shape #dataset

(torch.Size([228146, 3]), torch.Size([228146]))

In [311]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [312]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [313]:
emb = C[X] # (32, 3, 10)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 200)
logits = h @ W2 + b2 # (32, 27)
#counts = logits.exp() # this will give us the counts for each of the 27 possible characters, we can then use these counts to calculate the probabilities and do backpropagation to update the weights and biases of the network. The shape of counts will be (number of examples, number of classes), in this case (number of examples, 27) because we have 27 possible characters to predict. We can then use these counts to calculate the probabilities and do backpropagation to update the weights and biases of the network.
#prob = counts/counts.sum(1, keepdims=True) # this will give us the
#loss = -prob[torch.arange(32), Y].log().mean() # this will give us the average negative log likelihood loss for the correct characters, we can then use this loss to do backpropagation to update the weights and biases of the network. The shape of this tensor will be a single scalar value because we are taking the mean of the negative log probabilities for all the examples. We can then use this loss to do backpropagation to update the weights and biases of the network.
F.cross_entropy(logits, Y) # this will give us the same loss as the code above, but it is more efficient because it does not require us to calculate the probabilities and then take the log and mean, it does all of that in one step. The shape of this tensor will be a single scalar value because we are taking the mean of the negative log probabilities for all the examples. We can then use this loss to do backpropagation to update the weights and biases of the network.
loss

tensor(2.3926, grad_fn=<NllLossBackward0>)

In [314]:
F.cross_entropy(logits, Y) # this will give us the same loss as the code above, but it is more efficient because it does not require us to calculate the probabilities and then take the log and mean, it does all of that in one step. The shape of this tensor will be a single scalar value because we are taking the mean of the negative log probabilities for all the examples. We can then use this loss to do backpropagation to update the weights and biases of the network.

tensor(19.5052)

In [315]:
logits = torch.tensor([-2,-3,0,5]) # (32, 27)
counts = logits.exp() # this will give us the counts for each of the 27 possible characters, we can then use these counts to calculate the probabilities and do backpropagation to update the weights and biases of the network. The shape of counts will be (number of examples, number of classes), in this case (number of examples, 27) because we have 27 possible characters to predict. We can then use these counts to calculate the probabilities and do backpropagation to update the weights and biases of the network.
probs = counts/counts.sum() # this will give us the probabilities for each of the 27 possible characters, we can then use these probabilities to calculate the loss and do backpropagation to update the weights and biases of the network. The shape of prob will be (number of examples, number of classes), in this case (number of examples, 27) because we have 27 possible characters to predict. We can then use these probabilities to calculate the loss and do backpropagation to update the weights and biases of the network. We need to sum the counts along the second dimension (the classes) to get the total count for each example, and then we need to keep the dimensions of the result so that we can divide the counts by the total count for each example to get the probabilities. This is done by setting keepdim=True in the sum function.     
probs


tensor([9.0466e-04, 3.3281e-04, 6.6846e-03, 9.9208e-01])

In [316]:
logits = torch.tensor([-100,-3,0,5]) # (32, 27)
counts = logits.exp() # this will give us the counts for each of the 27 possible characters, we can then use these counts to calculate the probabilities and do backpropagation to update the weights and biases of the network. The shape of counts will be (number of examples, number of classes), in this case (number of examples, 27) because we have 27 possible characters to predict. We can then use these counts to calculate the probabilities and do backpropagation to update the weights and biases of the network.
probs = counts/counts.sum() # this will give us the probabilities for each of the 27 possible characters, we can then use these probabilities to calculate the loss and do backpropagation to update the weights and biases of the network. The shape of prob will be (number of examples, number of classes), in this case (number of examples, 27) because we have 27 possible characters to predict. We can then use these probabilities to calculate the loss and do backpropagation to update the weights and biases of the network. We need to sum the counts along the second dimension (the classes) to get the total count for each example, and then we need to keep the dimensions of the result so that we can divide the counts by the total count for each example to get the probabilities. This is done by setting keepdim=True in the sum function.     
probs

tensor([0.0000e+00, 3.3311e-04, 6.6906e-03, 9.9298e-01])

In [317]:
logits = torch.tensor([-2,-3,0,100]) # (32, 27)
counts = logits.exp() # this will give us the counts for each of the 27 possible characters, we can then use these counts to calculate the probabilities and do backpropagation to update the weights and biases of the network. The shape of counts will be (number of examples, number of classes), in this case (number of examples, 27) because we have 27 possible characters to predict. We can then use these counts to calculate the probabilities and do backpropagation to update the weights and biases of the network.
probs = counts/counts.sum() # this will give us the probabilities for each of the 27 possible characters, we can then use these probabilities to calculate the loss and do backpropagation to update the weights and biases of the network. The shape of prob will be (number of examples, number of classes), in this case (number of examples, 27) because we have 27 possible characters to predict. We can then use these probabilities to calculate the loss and do backpropagation to update the weights and biases of the network. We need to sum the counts along the second dimension (the classes) to get the total count for each example, and then we need to keep the dimensions of the result so that we can divide the counts by the total count for each example to get the probabilities. This is done by setting keepdim=True in the sum function.     
probs


tensor([0., 0., 0., nan])

In [318]:
counts
#very positive nu8mber gives us a number that os oput of range, there fore we cannot pass very large numbers

tensor([0.1353, 0.0498, 1.0000,    inf])

In [319]:
for p in parameters:
    p.requires_grad=True

In [320]:
lre =torch.linspace(-3,0,1000)
lrs = 10**lre
lrs

tensor([0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0011,
        0.0011, 0.0011, 0.0011, 0.0011, 0.0011, 0.0011, 0.0011, 0.0011, 0.0011,
        0.0011, 0.0011, 0.0011, 0.0012, 0.0012, 0.0012, 0.0012, 0.0012, 0.0012,
        0.0012, 0.0012, 0.0012, 0.0012, 0.0012, 0.0012, 0.0013, 0.0013, 0.0013,
        0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0014,
        0.0014, 0.0014, 0.0014, 0.0014, 0.0014, 0.0014, 0.0014, 0.0014, 0.0014,
        0.0015, 0.0015, 0.0015, 0.0015, 0.0015, 0.0015, 0.0015, 0.0015, 0.0015,
        0.0015, 0.0016, 0.0016, 0.0016, 0.0016, 0.0016, 0.0016, 0.0016, 0.0016,
        0.0016, 0.0017, 0.0017, 0.0017, 0.0017, 0.0017, 0.0017, 0.0017, 0.0017,
        0.0018, 0.0018, 0.0018, 0.0018, 0.0018, 0.0018, 0.0018, 0.0018, 0.0019,
        0.0019, 0.0019, 0.0019, 0.0019, 0.0019, 0.0019, 0.0019, 0.0020, 0.0020,
        0.0020, 0.0020, 0.0020, 0.0020, 0.0020, 0.0021, 0.0021, 0.0021, 0.0021,
        0.0021, 0.0021, 0.0021, 0.0022, 

In [325]:
lri = []
lossi = []

for i in range(10000):
    
    # minibatch contruct
    ix = torch.randint(0,X.shape[0],(32,)) # generate a random batch of 32 examples from the dataset, we can then use this batch to do the forward pass and calculate the loss and do backpropagation to update the weights and biases of the network. The shape of ix will be (32,) because we are selecting 32 random examples from the dataset. We can then use this ix to index into X and Y to get the corresponding input and target for each example in the batch.
    
    #Forward pass
    emb = C[X[ix]] # (32, 3, 10)
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 200)
    logits = h @ W2 + b2 # (32, 27)
    loss = F.cross_entropy(logits, Y[ix])
    print(loss.item())

    
    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    

    
    #update 
    lr = 0.1 # learning rate
    for p in parameters:
        p.data +=-lr * p.grad


    #track stats
    #lri.append(lre[i])
    #lossi.append(loss.item())

print(loss.item())






2.112537145614624
2.4755890369415283
2.504531145095825
2.7113800048828125
2.736325979232788
2.756141185760498
2.6279969215393066
2.225416660308838
2.878005027770996
2.6993539333343506
2.314929962158203
2.104372978210449
2.465219259262085
2.1530613899230957
2.359048843383789
2.1588456630706787
2.659668207168579
2.7185773849487305
2.54640793800354
2.748491048812866
2.114119529724121
2.952345848083496
2.3090202808380127
2.4170148372650146
2.413604497909546
2.3978466987609863
2.6421499252319336
2.2126433849334717
2.4207441806793213
2.4321935176849365
2.052952289581299
2.6968958377838135
2.513129711151123
2.494738817214966
2.4976089000701904
2.3237340450286865
2.7314960956573486
2.2718799114227295
2.714968681335449
2.592694044113159
2.5798442363739014
2.4120895862579346
2.58194637298584
2.553133010864258
2.318345069885254
2.805232286453247
2.6398634910583496
2.4087772369384766
2.633443593978882
2.5645930767059326
2.5287036895751953
2.215811014175415
2.7548787593841553
2.101114273071289
1.79

In [131]:
logits.max(1)

torch.return_types.max(
values=tensor([11.9629, 14.9534, 19.4986, 18.8153, 14.6295, 11.9629, 14.1185, 12.6405,
        14.3367, 16.3292, 13.8440, 18.8811, 11.9629, 14.6526, 15.2419, 18.0184,
        11.9629, 14.7612, 12.8592, 14.7470, 16.8057, 13.7583,  9.0044,  8.9571,
        14.4153, 11.9629, 14.4762, 15.0232, 11.6543, 14.9497, 16.9366, 13.5129],
       grad_fn=<MaxBackward0>),
indices=tensor([ 1, 13, 13,  1,  0,  1, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  1, 19,
         1,  2,  5, 12, 12,  1,  0,  1, 15, 16,  8,  9,  1,  0]))

In [132]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [199]:
torch.randint(0,X.shape[0],(32,))

tensor([132933,  55347,  42015,  21700, 217805,  69642, 123869, 129390, 179709,
         94695, 190453, 120304, 200019, 208846, 180526, 177763,  48114, 226322,
         58460, 142442, 156291,  25134, 216300, 105067, 142341, 192475, 155318,
         24618,  88662, 206759,  47966, 223435])

In [326]:
 #Forward pass
emb = C[X[ix]] # (32, 3, 10)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 200)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Y[ix])
loss

tensor(2.0651, grad_fn=<NllLossBackward0>)

To prevent model memorization as network grows, we split the dataset into 3
Training
Dev/validation set
Test set
Typically 80/10/10
80% used to optimize the parameters of hte model
10% used over the development of the hyperparameters of hte model
10% is for evaluating the performance of the model